# DepthWizard — SAM Building Segmentation Benchmark

**Question this notebook answers:** should SAM replace the hand-tuned CV segmentation in `segmentation.py`?

**Why we're asking:** the local CV segmentation was measured to mark roof *edges* rather than roof *interiors* — its primary building signal is edge density. Everything downstream inherits fragments, and morphology can't reconstruct regions that were never detected.

**How we decide:** DFC2019 CLS rasters carry real LAS-spec labels (class 6 = Buildings). Both approaches are scored against that same ground truth. If SAM doesn't win, we don't swap.

---

### Before running
1. `Runtime → Change runtime type → T4 GPU`
2. Upload `colab_subset.zip` (0.50 GB) to your Google Drive root

## 1. Verify GPU

In [ ]:
!nvidia-smi
import torch
print('cuda:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit('No GPU. Runtime > Change runtime type > T4 GPU, then rerun.')

## 2. Dependencies + SAM checkpoint

In [ ]:
!pip -q install segment-anything rasterio
# ViT-B, 375 MB. ViT-H is more accurate but ~6x slower; start here.
!wget -q -nc https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth
!ls -lh sam_vit_b_01ec64.pth

## 3. Mount Drive and unpack the data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!test -f /content/drive/MyDrive/colab_subset.zip || echo 'MISSING: upload colab_subset.zip to Drive root'
!mkdir -p /content/colab_subset
!unzip -q -o /content/drive/MyDrive/colab_subset.zip -d /content/colab_subset
!echo 'rgb tiles:'   && ls /content/colab_subset/rgb   | wc -l
!echo 'truth files:' && ls /content/colab_subset/truth | wc -l

## 4. Benchmark script

Upload `depthwizard_sam_benchmark.py` alongside this notebook, or paste it into a cell. It contains the SAM mask generation, the evidence-based building classification, and the metric functions.

In [ ]:
from google.colab import files
import os
if not os.path.exists('depthwizard_sam_benchmark.py'):
    print('Select depthwizard_sam_benchmark.py')
    files.upload()

## 5. Sanity check on one tile before the full run

In [ ]:
import os
os.environ['DW_DATA'] = '/content/colab_subset'

import importlib, depthwizard_sam_benchmark as bench
importlib.reload(bench)

bench.run(max_tiles=1)

## 6. Visual check

Numbers can look fine while the masks are wrong. Look before trusting the table.

In [ ]:
import numpy as np, cv2, glob, rasterio
import matplotlib.pyplot as plt

tile = sorted({os.path.basename(p)[:7] for p in glob.glob('/content/colab_subset/truth/*_CLS.tif')})[0]
rgb_path = sorted(glob.glob(f'/content/colab_subset/rgb/{tile}_*_RGB.tif'))[0]

with rasterio.open(rgb_path) as s: rgb_full = np.transpose(s.read([1,2,3]), (1,2,0))
with rasterio.open(f'/content/colab_subset/truth/{tile}_CLS.tif') as s: cls = s.read(1)
with rasterio.open(f'/content/colab_subset/truth/{tile}_DSM.tif') as s: dsm = s.read(1).astype(np.float32)

rgb = cv2.resize(rgb_full, (512,512), interpolation=cv2.INTER_AREA)
gt  = (cls == 6)

gen = bench.load_sam()
masks = gen.generate(rgb)
scored = bench.classify_masks_as_buildings(masks, rgb, dsm)

sam_pred = np.zeros((512,512), bool)
for m in scored:
    if m['score'] >= 0.35: sam_pred |= m['segmentation']
base_pred = bench.baseline_building_mask(rgb)

def overlay(img, mask, colour):
    o = img.copy(); o[mask] = colour
    return (0.55*img + 0.45*o).astype(np.uint8)

fig, ax = plt.subplots(1, 4, figsize=(22,6))
for a, im, t in zip(ax,
    [rgb, overlay(rgb, gt, [255,0,0]), overlay(rgb, sam_pred, [0,255,0]), overlay(rgb, base_pred, [255,160,0])],
    ['RGB', 'GROUND TRUTH (LAS class 6)', f'SAM ({len(masks)} masks)', 'baseline CV']):
    a.imshow(im); a.set_title(t); a.axis('off')
plt.tight_layout(); plt.show()

## 7. Full benchmark

Roughly 10–30 s per tile on a T4.

In [ ]:
bench.run(max_tiles=20)

## 8. Threshold sweep

The building/not-building score cutoff trades precision against recall. Pick it from the curve rather than by feel.

In [ ]:
valid = (cls != 65)
print(f"{'thresh':>7}{'kept':>7}{'IoU':>8}{'P':>8}{'R':>8}{'F1':>8}")
for t in [0.20,0.25,0.30,0.35,0.40,0.45,0.50,0.55,0.60]:
    p = np.zeros((512,512), bool); kept = 0
    for m in scored:
        if m['score'] >= t: p |= m['segmentation']; kept += 1
    mt = bench.mask_metrics(p, gt, valid)
    print(f"{t:>7.2f}{kept:>7d}{mt['iou']:>8.3f}{mt['precision']:>8.3f}{mt['recall']:>8.3f}{mt['f1']:>8.3f}")

## 9. Save results back to Drive

In [ ]:
!cp /content/colab_subset/sam_benchmark_results.json /content/drive/MyDrive/ 2>/dev/null && echo 'saved to Drive' || echo 'run section 7 first'

---
## What to send back

1. The **aggregate table** from section 7 (baseline vs SAM: IoU / precision / recall / F1)
2. The **per-building recall** rows — this is the "no building left behind" number
3. The **4-panel image** from section 6
4. The **threshold sweep** from section 8

If SAM wins on F1 and per-building recall, we swap `segmentation.py` and rerun the DFC2019 DSM benchmark to confirm the accuracy numbers move the right way.

If it doesn't win, that result stands and we investigate rather than swapping on faith.